# Week 11 — Notebook 4: Hierarchical Clustering

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Implement agglomerative clustering from scratch using single, complete, and average linkage
2. Build and interpret a dendrogram — including how to read the height and identify the natural number of clusters
3. Cut the dendrogram at different heights to obtain different values of k
4. Compare the four main linkage criteria empirically on both synthetic and retail data

---

**Theme:** Retail customer segmentation — using hierarchical clustering to find natural groupings without specifying k upfront.


## The Family Tree Analogy

Imagine a genealogist reconstructing family trees for a village of 20 strangers. She only has a table of pairwise similarity scores — how much each pair of people resembles each other in looks, habits, and speech.

She starts by declaring **every person their own family** (20 families of 1). Then she repeatedly finds the **two most similar families** and merges them into one. First she might merge two brothers. Then she merges those brothers with their cousin. Then that extended family merges with a related clan from the next village.

She continues until everyone is in a single giant family. The result is a **family tree** — a hierarchy that records every merger, in order, along with how similar the merged groups were.

That family tree is exactly what a **dendrogram** shows:
- The **x-axis** lists all the original people (data points)
- The **y-axis** is the similarity (or distance) at which each merger happened
- A **horizontal bar** high up means two very different groups were merged late — a big gap between that bar and the one below it suggests a natural boundary between clusters

Unlike K-means, you never have to tell the genealogist how many families to return — she gives you the full tree and you decide later at which generation to cut it.


## Why This Matters for ML

Hierarchical clustering addresses three core weaknesses of K-means:

| Feature | K-Means | Hierarchical |
|---|---|---|
| Must specify k before running | ✅ Yes (limitation) | ❌ No — choose k after seeing the tree |
| Reveals multi-scale structure | ❌ No | ✅ Yes — all levels visible in one dendrogram |
| Human-interpretable output | Cluster centroids only | Full merge history + dendrogram |
| Sensitive to outliers | High | Moderate (depends on linkage) |
| Scalable to millions of rows | ✅ Yes | ❌ No — O(n²) to O(n³) |

**Retail applications:**
- Customer segmentation when you do not know how many segments exist
- Product taxonomy: group SKUs into a hierarchy (subcategory → category → department)
- Geographic analysis: cluster stores by performance metrics and visualise the hierarchy


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time

from scipy.cluster.hierarchy import linkage, dendrogram, cophenet, fcluster
from scipy.spatial.distance  import pdist, squareform

from sklearn.cluster          import AgglomerativeClustering
from sklearn.datasets         import make_blobs
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics          import adjusted_rand_score

np.random.seed(42)                    # global reproducibility
warnings.filterwarnings('ignore')     # suppress scipy/sklearn minor warnings
sns.set_theme(style='whitegrid')      # consistent seaborn style

print('All imports successful.')


---
## Two Flavours of Hierarchical Clustering

### Agglomerative (bottom-up) — the standard approach

Start with **n clusters** (every point alone) and **merge** the two most similar clusters at each step until only one cluster remains.

- Pro: efficient merge operations, well-studied linkage criteria
- Used by: scipy, sklearn, most ML libraries

### Divisive (top-down) — rarely used in practice

Start with **1 cluster** (all points together) and **split** the least cohesive cluster at each step.

- Pro: can be more natural for some hierarchies (e.g., biological taxonomy)
- Con: requires solving a hard sub-problem at each split; computationally expensive
- Rarely implemented in standard ML libraries

**We focus on agglomerative clustering** throughout this notebook.


In [ ]:
def agglomerative_scratch(X, linkage_type='single'):
    """
    From-scratch agglomerative clustering.

    Parameters
    ----------
    X            : (n, d) array of data points
    linkage_type : 'single' | 'complete' | 'average'

    Returns
    -------
    merge_history : list of (cluster_a_id, cluster_b_id, merge_distance, new_size)
                    One entry per merge step (n-1 total).
    """
    n = len(X)

    # ── Step 1: compute full pairwise distance matrix ─────────────────────────
    dist = np.zeros((n, n))                                   # n×n distance matrix
    for i in range(n):
        for j in range(i + 1, n):
            d = np.linalg.norm(X[i] - X[j])                  # Euclidean distance
            dist[i, j] = dist[j, i] = d                      # symmetric

    # ── Step 2: each point starts in its own cluster ──────────────────────────
    # clusters maps cluster_id -> list of original point indices
    clusters      = {i: [i] for i in range(n)}               # {0:[0], 1:[1], ..., n-1:[n-1]}
    merge_history = []                                        # will hold one tuple per merge
    active        = list(range(n))                            # ids of currently-live clusters
    next_id       = n                                         # id for the next merged cluster

    # ── Step 3: repeatedly merge the two closest clusters ────────────────────
    while len(active) > 1:
        min_dist  = np.inf
        merge_a   = -1
        merge_b   = -1

        # Compare every pair of active clusters
        for i in range(len(active)):
            for j in range(i + 1, len(active)):
                ci = active[i]                                # id of first cluster
                cj = active[j]                               # id of second cluster
                pts_i = clusters[ci]                         # point indices in cluster ci
                pts_j = clusters[cj]                         # point indices in cluster cj

                # Compute all pairwise distances between the two clusters
                pairwise = [dist[a, b] for a in pts_i for b in pts_j]

                # Apply the chosen linkage rule
                if linkage_type == 'single':                  # nearest-neighbour distance
                    d = min(pairwise)
                elif linkage_type == 'complete':              # furthest-neighbour distance
                    d = max(pairwise)
                elif linkage_type == 'average':               # mean of all pairwise distances
                    d = np.mean(pairwise)
                else:
                    raise ValueError(f'Unknown linkage: {linkage_type}')

                if d < min_dist:
                    min_dist = d
                    merge_a  = ci
                    merge_b  = cj

        # Perform the merge: create a new cluster containing all points from both
        new_size                  = len(clusters[merge_a]) + len(clusters[merge_b])
        clusters[next_id]         = clusters[merge_a] + clusters[merge_b]  # union of point lists
        merge_history.append((merge_a, merge_b, min_dist, new_size))       # record the event

        active.remove(merge_a)   # retire the two merged clusters
        active.remove(merge_b)
        active.append(next_id)   # add the new combined cluster
        next_id += 1             # increment id counter for next merge

    return merge_history


print('agglomerative_scratch() defined — supports single, complete, and average linkage.')


In [ ]:
np.random.seed(42)

# ── Create 8 easy-to-follow 2-D points ────────────────────────────────────────
X8 = np.array([
    [1.0, 1.0],   # point 0
    [1.5, 1.2],   # point 1  (close to 0)
    [5.0, 5.0],   # point 2
    [5.2, 4.8],   # point 3  (close to 2)
    [9.0, 1.0],   # point 4
    [9.3, 1.4],   # point 5  (close to 4)
    [3.0, 7.0],   # point 6
    [2.8, 6.7],   # point 7  (close to 6)
])

# ── Run our scratch implementation with all three linkages ────────────────────
for lt in ['single', 'complete', 'average']:
    history = agglomerative_scratch(X8, linkage_type=lt)
    print(f'\n=== {lt.upper()} LINKAGE merge history ===')
    print(f'{"Step":<5} {"Merge":<20} {"Distance":>10}  {"New size":>10}')
    print('-' * 50)
    for step, (a, b, d, size) in enumerate(history, start=1):
        # Show the original point ids or cluster ids that were merged
        print(f'{step:<5} cluster {a:>3} + cluster {b:>3}  {d:>10.4f}  {size:>10}')


---
## The Dendrogram

A **dendrogram** is a tree diagram that visualises the complete merge history from agglomerative clustering:

- **x-axis:** the original data points (or leaves of the tree)
- **y-axis:** the distance (or dissimilarity) at which each merge occurred
- **Each horizontal bar** connects two clusters that were merged; the **height of the bar = distance at merge time**

### How to read a dendrogram

1. **A short bar** means two very similar things were merged — the merge happened early and the clusters are tight
2. **A tall bar** means two dissimilar groups were finally joined — there is a large gap between them
3. **The natural number of clusters** corresponds to the level just before the **biggest vertical jump** in bar heights: draw a horizontal line through that gap, count how many branches it cuts — that is your k

The dashed line in the plot below marks the cut for k = 4 clusters.


In [ ]:
np.random.seed(42)

# ── Use scipy's linkage on the 8-point dataset ───────────────────────────────
# scipy's linkage() returns a (n-1, 4) matrix: [cluster_a, cluster_b, distance, size]
Z8 = linkage(X8, method='single', metric='euclidean')

# ── Plot the dendrogram and annotate the cut for k=4 ─────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

dend = dendrogram(
    Z8,
    ax=ax,
    labels=[f'P{i}' for i in range(8)],   # label each leaf with its point index
    leaf_rotation=0,
    leaf_font_size=11,
    color_threshold=3.5,                   # colour branches below this height
)

# Draw a horizontal cut line at height ~3.5 → yields 4 clusters
ax.axhline(y=3.5, color='crimson', linestyle='--', linewidth=2,
           label='Cut here → k = 4 clusters')
ax.set_xlabel('Data Points', fontsize=11)
ax.set_ylabel('Merge Distance (height)', fontsize=11)
ax.set_title('Dendrogram for 8 Points — Single Linkage\n'
             'Read from bottom up: short bars = tight local pairs; tall bars = big jumps',
             fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  P0+P1 merge first (very close in space), then P2+P3, then P4+P5, then P6+P7.')
print('  The big gap just below height 3.5 tells us the natural k is 4.')
print('  Cutting at the dashed line gives us 4 groups: {P0,P1}, {P2,P3}, {P4,P5}, {P6,P7}.')


---
## The Four Linkage Criteria — Explained

The **linkage criterion** defines how we measure the distance between two **clusters** (as opposed to two individual points). The choice has a large impact on the shape of the resulting clusters.

### Single linkage (minimum distance)
Distance between cluster A and cluster B = the **minimum** pairwise distance between any point in A and any point in B.

> d(A, B) = min{ dist(a, b) : a ∈ A, b ∈ B }

Effect: merges clusters that are **nearby in at least one point**. Prone to the **chaining effect** — a long chain of individually close points can be merged into one giant cluster even if the endpoints are far apart.

### Complete linkage (maximum distance)
Distance = the **maximum** pairwise distance between any point in A and any point in B.

> d(A, B) = max{ dist(a, b) : a ∈ A, b ∈ B }

Effect: only merges two clusters if **all their points** are close enough. Produces **compact, tightly bounded** clusters. Sensitive to outliers (one extreme point inflates the maximum).

### Average linkage (mean distance)
Distance = the **mean** of all pairwise distances between A and B.

> d(A, B) = (1 / |A||B|) · Σₐ Σᵦ dist(a, b)

Effect: a balance between single and complete. Moderately robust to outliers. Often a good default.

### Ward linkage (variance minimisation)
Distance = the **increase in total within-cluster variance** that would result from merging A and B.

> d(A, B) = [ (|A||B|) / (|A|+|B|) ] · ‖μ_A − μ_B‖²

Effect: most similar to K-means — produces roughly equal-sized, compact clusters. Usually the **best default choice** for exploratory analysis.


In [ ]:
np.random.seed(42)

# ── Dataset: 4 well-separated blobs ──────────────────────────────────────────
X4, y4 = make_blobs(n_samples=120, centers=4, cluster_std=0.7, random_state=42)

linkage_methods = ['single', 'complete', 'average', 'ward']

# ── Compute scipy linkage matrices for each method ───────────────────────────
Z_dict = {}
for method in linkage_methods:
    Z_dict[method] = linkage(X4, method=method, metric='euclidean')

# ── Plot four dendrograms side by side ────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

descriptions = {
    'single':   'Chaining effect\n(long snaky clusters)',
    'complete': 'Compact clusters\n(sensitive to outliers)',
    'average':  'Balanced\n(good default)',
    'ward':     'Variance-minimising\n(most like K-Means)',
}

for ax, method in zip(axes, linkage_methods):
    dendrogram(
        Z_dict[method],
        ax=ax,
        no_labels=True,            # hide individual point labels (too many)
        color_threshold=0.7 * max(Z_dict[method][:, 2]),   # colour threshold at 70% of max height
    )
    ax.set_title(f'{method.capitalize()} Linkage\n{descriptions[method]}',
                 fontsize=10)
    ax.set_ylabel('Merge Distance', fontsize=9)
    ax.tick_params(axis='x', which='both', bottom=False)   # remove x-tick marks

fig.suptitle('Four Linkage Criteria on the Same 4-Blob Dataset\n'
             'Notice: single linkage chains; Ward produces the clearest "gaps"',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

# ── Cut each dendrogram to get k=4 clusters and compare to ground truth ───────
print('Adjusted Rand Index (ARI) for k=4 cut — higher is better (max=1.0):')
print('-' * 45)

ari_scores = {}
for method in linkage_methods:
    # fcluster cuts the dendrogram to produce flat cluster labels
    labels_cut = fcluster(Z_dict[method], t=4, criterion='maxclust')   # t=4 → exactly 4 clusters
    ari = adjusted_rand_score(y4, labels_cut)                          # compare with true labels
    ari_scores[method] = ari
    print(f'  {method.capitalize():<10}: ARI = {ari:.4f}')

# ── Bar chart of ARI scores ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors  = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
bars    = ax.bar(list(ari_scores.keys()),
                 list(ari_scores.values()),
                 color=colors, width=0.5, edgecolor='black')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Adjusted Rand Index (ARI)', fontsize=11)
ax.set_title('ARI vs. Ground Truth (k=4) — Which Linkage Recovers the True Clusters?',
             fontsize=11)
for bar, val in zip(bars, ari_scores.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

best_method = max(ari_scores, key=ari_scores.get)
print(f'\nBest linkage for this dataset: {best_method} (ARI = {ari_scores[best_method]:.4f})')


---
## Cophenetic Correlation: How Faithful Is the Dendrogram?

The **cophenetic correlation coefficient** answers the question: *how well does the dendrogram preserve the original pairwise distances?*

Formally:
> cophenetic_distance(i, j) = the height at which points i and j first appear in the same cluster (i.e., the height of their lowest common ancestor in the tree)

> cophenetic correlation = Pearson correlation between the original pairwise distances and the cophenetic distances

Range: typically 0 to 1. **Higher = the dendrogram is a more faithful representation of the data's distance structure.**

- Values above **0.75** are generally considered a good fit
- A low value means the dendrogram is distorting the true distances — the hierarchical structure may not be meaningful


In [ ]:
np.random.seed(42)

# ── Compute cophenetic correlation for each linkage ───────────────────────────
# cophenet() returns (correlation, cophenetic_distances_vector)
coph_scores = {}
for method in linkage_methods:
    c, _coph_dists = cophenet(Z_dict[method], pdist(X4, metric='euclidean'))
    coph_scores[method] = c
    print(f'  {method.capitalize():<10}: cophenetic r = {c:.4f}')

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors  = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
bars    = ax.bar(list(coph_scores.keys()),
                 list(coph_scores.values()),
                 color=colors, width=0.5, edgecolor='black')
ax.axhline(y=0.75, color='black', linestyle='--', linewidth=1.5,
           label='Acceptable threshold (0.75)')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Cophenetic Correlation Coefficient', fontsize=11)
ax.set_title('Cophenetic Correlation — How Faithfully Does Each Dendrogram\nPreserve Pairwise Distances?',
             fontsize=11)
ax.legend(fontsize=10)
for bar, val in zip(bars, coph_scores.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

best_coph = max(coph_scores, key=coph_scores.get)
print(f'\nHighest cophenetic correlation: {best_coph} linkage ({coph_scores[best_coph]:.4f})')
print('Higher cophenetic correlation means the dendrogram is a more faithful summary of the data.')


---
## How to Read a Dendrogram: Finding the Natural k

The practical question when looking at a dendrogram is: **where do I cut?**

The answer lies in the **gap structure** of the y-axis:

1. Look at the last few merges before everything collapses to one cluster (the top of the tree)
2. Find the **biggest vertical gap** — the largest jump in height between consecutive merge events
3. Draw a **horizontal line through that gap**
4. Count the number of branches the line crosses — that is your natural k

**Analogy:** imagine driving uphill and looking for the steepest section of road. The steepest section (biggest height gain per meter) tells you where nature put the sharpest boundary.

If there is no clear gap, the data may not have a clean hierarchical structure — or there may be many equally valid levels of granularity.


In [ ]:
np.random.seed(42)

# ── Synthetic retail data: TotalSpend, Frequency, Recency ────────────────────
# Simulate 3 natural customer segments with distinct profiles
n_per_segment = 50

# Segment 1: High-spend, high-frequency, recent (VIP)
seg1 = np.column_stack([
    np.random.normal(2000, 200, n_per_segment),   # TotalSpend (£)
    np.random.normal(12,   1.5, n_per_segment),   # Frequency (visits/month)
    np.random.normal(2,    0.5, n_per_segment),   # Recency (days since last visit)
])

# Segment 2: Medium-spend, medium-frequency (regular)
seg2 = np.column_stack([
    np.random.normal(500,  80,  n_per_segment),
    np.random.normal(5,    1.0, n_per_segment),
    np.random.normal(10,   2.0, n_per_segment),
])

# Segment 3: Low-spend, infrequent, lapsed (churned)
seg3 = np.column_stack([
    np.random.normal(80,   20,  n_per_segment),
    np.random.normal(1,    0.3, n_per_segment),
    np.random.normal(60,   10,  n_per_segment),
])

X_retail = np.vstack([seg1, seg2, seg3])            # (150, 3) matrix
y_retail  = np.array([0]*n_per_segment + [1]*n_per_segment + [2]*n_per_segment)

# ── Scale before clustering (features have very different units) ──────────────
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_retail)

# ── Ward linkage dendrogram ───────────────────────────────────────────────────
Z_retail  = linkage(X_scaled, method='ward')

# Find the biggest gap in merge heights to identify the natural k
heights   = Z_retail[:, 2]                          # merge distances (sorted ascending)
gaps      = np.diff(heights)                        # differences between consecutive merges
biggest_gap_idx = np.argmax(gaps)                   # index of the largest gap
cut_height      = (heights[biggest_gap_idx] + heights[biggest_gap_idx + 1]) / 2  # midpoint

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    Z_retail,
    ax=ax,
    no_labels=True,
    color_threshold=cut_height,
)
ax.axhline(y=cut_height, color='crimson', linestyle='--', linewidth=2.5,
           label=f'Natural cut height = {cut_height:.2f}')

# Annotate the gap
ax.annotate('Biggest gap\n→ natural k = 3',
            xy=(ax.get_xlim()[1] * 0.85, cut_height),
            fontsize=11, color='crimson',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='crimson', alpha=0.8))

ax.set_xlabel('Customers (150 total)', fontsize=11)
ax.set_ylabel('Ward Merge Distance', fontsize=11)
ax.set_title('Retail Customer Dendrogram (Ward Linkage)\nBiggest gap reveals k = 3 natural segments',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Biggest gap found at height {cut_height:.3f}')
print(f'Cutting at this height gives k = 3 clusters — matching the true 3 segments.')


In [ ]:
np.random.seed(42)

# ── Cut the dendrogram at the natural height to get 3 flat cluster labels ────
labels_retail = fcluster(Z_retail, t=3, criterion='maxclust')   # exactly 3 clusters
labels_retail -= 1                                               # convert 1-indexed → 0-indexed

ari_retail = adjusted_rand_score(y_retail, labels_retail)
print(f'ARI vs. ground truth: {ari_retail:.4f}  (1.0 = perfect recovery)')

# ── Visualise segments in 2D (TotalSpend vs Frequency, scaled) ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette   = ['#e41a1c', '#377eb8', '#4daf4a']
seg_names = ['VIP', 'Regular', 'Churned']

for ax, (labels, title) in zip(axes, [
        (y_retail,      'Ground Truth Segments'),
        (labels_retail, f'Hierarchical Clustering (ARI={ari_retail:.3f})')]):

    for k, (col, name) in enumerate(zip(palette, seg_names)):
        mask = labels == k
        ax.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                   c=col, s=30, alpha=0.75, label=name)

    ax.set_xlabel('Scaled Total Spend', fontsize=10)
    ax.set_ylabel('Scaled Frequency', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

fig.suptitle('Retail Customer Segments — Ward Hierarchical Clustering',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Print segment profiles in original feature space ────────────────────────
df_retail = pd.DataFrame(X_retail, columns=['TotalSpend', 'Frequency', 'Recency'])
df_retail['Segment'] = labels_retail
print('\nSegment profiles (original feature space):')
print(df_retail.groupby('Segment')[['TotalSpend', 'Frequency', 'Recency']].mean().round(1))


---
## Time Complexity

### Why hierarchical clustering doesn't scale to large datasets

Naïve agglomerative clustering:
- **Space:** O(n²) — must store the full pairwise distance matrix
- **Time:** O(n³) — for each of the n-1 merges, scan all O(n²) remaining pairs

Optimised implementations (e.g., with priority queues and Lance-Williams update formulas):
- **Time:** O(n² log n) — still quadratic space and super-linear time

Compare with **K-means:** O(n · k · d · i) where k = clusters, d = dimensions, i = iterations. This is effectively **linear in n** for fixed k, d, i.

**Practical guideline:**
- n < 5,000: hierarchical clustering is comfortable
- 5,000 < n < 50,000: use `AgglomerativeClustering` with `connectivity` constraints to speed up
- n > 50,000: prefer K-means, DBSCAN, or approximate nearest-neighbour methods


In [ ]:
np.random.seed(42)

# ── Benchmark AgglomerativeClustering on increasing n ─────────────────────────
n_values  = [100, 500, 1000, 2000]
times_agg = []

for n in n_values:
    X_bench, _ = make_blobs(n_samples=n, centers=4, random_state=42)   # fresh data per size
    start = time.perf_counter()                                         # high-resolution timer
    AgglomerativeClustering(n_clusters=4, linkage='ward').fit(X_bench)
    elapsed = time.perf_counter() - start
    times_agg.append(elapsed)
    print(f'  n = {n:>5}:  {elapsed:.4f}s')

# ── Fit a power-law curve to show O(n^p) scaling ────────────────────────────
log_n     = np.log(n_values)
log_t     = np.log(times_agg)
coeffs    = np.polyfit(log_n, log_t, deg=1)    # linear fit in log-log space → exponent
exponent  = coeffs[0]                          # slope = empirical scaling exponent

n_fit     = np.linspace(100, 2200, 200)
t_fit     = np.exp(np.polyval(coeffs, np.log(n_fit)))   # fitted curve

# ── Plot scaling curve ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(n_values, times_agg, color='steelblue', s=80, zorder=5, label='Measured time')
ax.plot(n_fit, t_fit, color='tomato', linewidth=2,
        label=f'Fitted O(n^{exponent:.2f})')
ax.set_xlabel('Number of samples (n)', fontsize=11)
ax.set_ylabel('Wall-clock time (seconds)', fontsize=11)
ax.set_title(f'AgglomerativeClustering Scaling\nEmpirical exponent ≈ {exponent:.2f} (theory: 2–3)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'\nEmpirical scaling exponent: n^{exponent:.2f}')
print('Even at n=2000 the runtime is manageable, but extrapolating to n=100,000 would be very slow.')


---
## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You look at a Ward linkage dendrogram for a customer dataset. The last four merge heights (from bottom to top) are: 0.8, 1.1, 1.3, 4.9. There is then one final merge at height 6.2. How many clusters does the data naturally contain, and how do you identify this from the dendrogram?

<details><summary>Show answer</summary>

The data naturally contains **2 clusters**.

To find this from the dendrogram:

1. Look at the **gaps between consecutive merge heights**: 1.1−0.8=0.3, 1.3−1.1=0.2, 4.9−1.3=**3.6**, 6.2−4.9=1.3
2. The **biggest gap** is **3.6 units** (between height 1.3 and height 4.9)
3. Draw a horizontal line through this gap (e.g., at height 3.1)
4. Count the branches the line crosses — it crosses **2 branches**, so k = 2

The large jump from 1.3 to 4.9 signals that the two sub-trees being merged at height 4.9 are genuinely different groups — the dendrogram is telling you to stop at k = 2.

</details>

---

**Question 2:** You run single linkage hierarchical clustering on a dataset and notice that the dendrogram shows one giant cluster absorbing individual points one at a time, rather than two compact subtrees merging. What is the name of this phenomenon, which linkage criteria suffer from it, and what linkage would you switch to?

<details><summary>Show answer</summary>

This is the **chaining effect** (also called chain-like or elongated clustering).

It occurs because **single linkage** measures distance between clusters as the **minimum** pairwise distance. If a long chain of points sits between two true clusters, each adjacent pair is very close, so they merge one by one — eventually connecting the two true clusters through the chain.

**Average linkage** also suffers mildly from chaining. **Complete linkage** and **Ward linkage** are much less prone to it because they require all (or most) points in two clusters to be close before merging.

**Switch to Ward linkage** — it minimises within-cluster variance and naturally produces compact, roughly equal-sized clusters with no chaining.

</details>

---

**Question 3:** A colleague says: "Hierarchical clustering is strictly better than K-means because you don't have to specify k." Give one argument in favour of this claim, and one important counter-argument.

<details><summary>Show answer</summary>

**In favour:** Hierarchical clustering genuinely does not require k upfront. Instead, you fit the model once and then inspect the dendrogram to choose k after seeing the data's structure. This is especially useful in exploratory analysis where you have no prior knowledge of the number of segments.

**Counter-argument:** Hierarchical clustering has **O(n²) space and O(n² log n) to O(n³) time complexity**, making it impractical for large datasets (say, n > 50,000). K-means runs in **O(n · k · d · i)** — effectively linear in n — so it scales to millions of customers. In most real-world retail applications the dataset is large and K-means (with a careful k-selection step) is far more practical. Also, if you already have strong domain knowledge about the number of segments, specifying k is not a burden — it is a feature.

</details>


---
## Key Takeaways

| Topic | Summary |
|---|---|
| Algorithm | Agglomerative: start with n clusters, merge the two closest at each step |
| Output | Dendrogram — full merge history, not just flat labels |
| Choosing k | Not required upfront; find the biggest gap in dendrogram heights |
| Single linkage | Minimum distance; fast but prone to chaining |
| Complete linkage | Maximum distance; compact clusters; sensitive to outliers |
| Average linkage | Mean distance; balanced; good default |
| Ward linkage | Variance minimisation; most like K-means; usually best default |
| Cophenetic r | Measures how faithfully dendrogram preserves original distances (>0.75 = good) |
| Time complexity | O(n² log n) to O(n³) — slow for large n |
| Best use cases | Exploratory analysis, small-to-medium datasets, hierarchical product/customer taxonomies |

**The bottom line:** Hierarchical clustering trades scalability for interpretability. The dendrogram gives a rich, human-readable summary of cluster structure at every level of granularity — something K-means can never provide. Use it when you want to **explore the data's natural hierarchy** without committing to a specific k upfront.

---
*Next: Notebook 05 — DBSCAN: Density-Based Clustering*
